# Jour 1 · Comprendre le RAG avant de coder


## Le problème de départ

Un modèle de langage sait produire du texte, mais il ne possède pas automatiquement nos procédures internes, leurs dernières modifications ni la preuve de l'endroit où une information a été trouvée.

Si nous demandons directement « Quel portail VPN devons-nous utiliser ? », il peut produire une adresse plausible sans connaître la nôtre.

Un **RAG** ajoute une étape de recherche avant la génération. Son nom signifie *Retrieval-Augmented Generation*, ou **génération augmentée par la recherche**.


![LLM seul et LLM alimenté par des documents](../assets/jour_01/00_llm_seul_vs_rag.png)

*Le RAG apporte au modèle des passages internes au moment de la question ; il ne modifie pas les poids du modèle.*


## Une définition à retenir

> Un RAG recherche d'abord des passages pertinents, puis les place dans le prompt pour aider le modèle à répondre.

**Pertinent** signifie « utile pour cette question ». Le passage le plus proche mathématiquement n'est pas automatiquement correct : nous apprendrons à contrôler cette recherche.

Le RAG ne garantit pas la vérité. Une erreur reste possible si le document manque, si son découpage est mauvais, si le retriever choisit mal ou si le modèle ajoute une information absente.


### En clair — un examen avec les documents ouverts

Un LLM seul essaie de répondre avec ce qu'il a appris auparavant. Un RAG lui donne d'abord les quelques pages utiles, comme si nous ouvrions le bon passage d'un manuel avant de répondre.

Le modèle n'est donc pas réentraîné et ne devient pas automatiquement plus intelligent : il reçoit simplement de meilleures informations pour cette question précise.


## Les deux pipelines

Un **pipeline** est une suite d'étapes où la sortie d'une étape devient l'entrée de la suivante.

- L'**ingestion** prépare les connaissances lorsque la documentation change.
- La **requête** traite chaque question utilisateur.


![Pipeline d'ingestion et pipeline de question](../assets/jour_01/00_deux_pipelines.png)

*Indexer n'est pas répondre : la première chaîne prépare la base, la seconde l'interroge.*


## Vocabulaire de départ

| Terme | Explication simple |
|---|---|
| document | un texte accompagné d'informations sur son origine |
| métadonnée | une étiquette comme le nom du fichier ou la page |
| chunk | un petit passage découpé dans un document |
| embedding | une liste de nombres représentant le sens d'un texte |
| vector store | une base qui stocke textes, vecteurs et métadonnées |
| Chroma | le vector store local utilisé dans ce cours |
| retriever | le composant qui récupère les passages candidats |
| top-k | les `k` meilleurs candidats conservés |
| contexte | les passages insérés dans le prompt |
| LLM | le modèle qui génère la réponse à partir du prompt |

Chaque terme sera repris avec un exemple avant d'apparaître dans une chaîne complète.


## Ce que le RAG n'est pas

- Ce n'est pas un réentraînement du modèle.
- Ce n'est pas une mémoire de conversation.
- Ce n'est pas automatiquement une recherche Internet.
- Ce n'est pas un agent qui choisit librement des actions.
- Ce n'est pas une garantie contre toutes les hallucinations.


### À vous de jouer — classer les étapes

Classez ces actions dans **avant les questions** ou **à chaque question** : charger un Markdown, calculer l'embedding de la question, découper un guide, rechercher les trois chunks les plus proches, appeler le LLM et stocker les vecteurs.


## Notre fil rouge

Nous construisons un assistant helpdesk à partir de cinq documents fictifs. La même question nous accompagnera :

> « Je travaille de chez moi. Quel portail dois-je saisir dans GlobalProtect ? »

Nous verrons successivement le passage retrouvé, le contexte exact, le prompt final, la réponse et sa source.


In [1]:
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Racine du cours :", ROOT)


Racine du cours : C:\Users\bstorm_user\Desktop\rag\cours_rag


In [2]:
files = sorted(path.name for path in DOCUMENTS_DIR.iterdir() if path.suffix in {".md", ".pdf"})
print("Documents disponibles :")
for filename in files:
    print("-", filename)


Documents disponibles :
- README.md
- guide_escalade_p1.pdf
- politique_securite.md
- procedure_ecran.md
- procedure_imprimante.md
- procedure_vpn.md


## À retenir

- Le RAG ajoute une recherche documentaire avant la génération.
- Ingestion et question sont deux pipelines distincts.
- La qualité finale dépend d'abord des passages retrouvés.
- Les sources viennent des métadonnées, pas de l'imagination du modèle.

```text
documents → chunks → embeddings → Chroma
question → retriever → contexte → LLM → réponse + sources
```
